# 02 - k-Nearest Neighbours

**Idea.** Classify each epoch by majority vote of its `k` closest neighbours in feature space ("similar physiology -> same stage").

**Good for this data:** captures nonlinearity, no training step, intuitive.

**Bad for this data:** distance-based, so it is very sensitive to feature scaling and to correlated/redundant features (they double-count in the distance). It has no native NaN support, so the half-missing column must be imputed - and imputed values distort neighbourhoods. 21 dimensions also start to thin out neighbourhoods (curse of dimensionality).

**Expected CV macro-F1:** ~0.764

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

# Notebooks live in day9/models/, data sits one level up in day9/
TRAIN_PATH = '../train.csv'
TEST_PATH  = '../test.csv'
OUT_DIR    = '../outputs'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# every column except the id and the label is a predictor
FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
TARGET   = 'sleep_stage'
y = train[TARGET]
print('train', train.shape, '| test', test.shape, '| #features', len(FEATURES))
train.head()

train (9000, 23) | test (5000, 22) | #features 21


,id,eeg_delta_power,eeg_theta_power,eeg_alpha_power,eeg_sigma_power,eeg_beta_power,eeg_gamma_power,eeg_slow_osc_power,eeg_spectral_entropy,eeg_spindle_density,...,eog_movement_density,eog_amplitude,heart_rate_mean,heart_rate_variability,respiration_rate,respiration_variability,spo2_mean,body_movement_index,eog_burst_index,sleep_stage
0,0,-1.51474,1.40728,10.33510,-1.61350,3.73081,0.99850,1.85689,-3.24253,-1.27096,...,2.65567,1.98733,1.60184,-2.49794,-0.59521,1.71154,1.93342,1.57365,-1.36230,1
1,1,-0.28998,0.89706,1.62494,2.41580,-2.70265,-0.10131,-1.68955,0.01442,-2.87943,...,4.36423,0.09942,3.38567,-0.56386,2.16016,-4.32301,1.07270,-2.43903,-0.37271,2
2,2,3.35435,0.32987,-5.41547,2.38680,-2.90584,-2.93372,-3.11713,-0.04647,1.61782,...,-3.87561,-0.87681,-2.84480,5.08383,-4.60411,0.37967,-2.06913,2.67324,NaN,3
3,3,-1.44917,-0.04374,1.71560,-1.27770,-0.19007,2.21826,1.69621,0.39756,0.00534,...,1.41415,0.39275,0.55060,-2.12910,2.32790,0.78319,0.98233,1.53824,-0.25040,1
4,4,1.35898,-2.36720,-7.40779,5.31815,-2.55954,-5.13284,-5.26634,1.73985,1.04618,...,-0.55616,0.86588,-1.96343,4.30036,0.22130,-1.44020,1.35760,-3.07701,-1.04947,3


## Preprocessing
**Scaling is mandatory** for kNN - without it, a feature with std 5 dominates the distance over one with std 1. We also median-impute the NaNs. Both go in a pipeline so they are fit per fold.

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier

clf = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=15),   # k=15 is a reasonable default; tune later
)
clf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive est

In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# SAME validation protocol for every model so scores are comparable:
#   - 5 folds, stratified  -> each fold keeps the 22-27% class balance
#   - shuffle + fixed seed  -> reproducible split
#   - scoring = macro-F1    -> the competition metric (every class weighted equally)
# cross_val_score trains on 4 folds and scores the held-out fold, 5x, and averages.
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def benchmark(model, X, label):
    s = cross_val_score(model, X, y, cv=cv, scoring='f1_macro')
    print(f'{label}: macro-F1 = {s.mean():.4f} +/- {s.std():.4f}')
    return s

## Benchmark

In [4]:
benchmark(clf, train[FEATURES], 'kNN (k=15)')

kNN (k=15): macro-F1 = 0.7641 +/- 0.0148


array([0.75206729, 0.74881262, 0.76601955, 0.79086046, 0.76271587])

## Quick look at the effect of k
A small sweep shows the bias/variance trade-off: tiny k overfits (noisy), large k oversmooths.

In [5]:
for k in [5, 15, 31, 51]:
    m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    benchmark(m, train[FEATURES], f'kNN k={k}')

kNN k=5: macro-F1 = 0.7349 +/- 0.0094


kNN k=15: macro-F1 = 0.7641 +/- 0.0148


kNN k=31: macro-F1 = 0.7626 +/- 0.0093


kNN k=51: macro-F1 = 0.7616 +/- 0.0093


## Create submission

In [6]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

# kNN needs imputation + scaling; pick k from the sweep above.
final_model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
final_model.fit(train[FEATURES], y)            # train on ALL 9000 labelled rows
pred = final_model.predict(test[FEATURES])     # predict the 5000 unlabelled test rows

submission = pd.DataFrame({'id': test['id'], 'sleep_stage': np.asarray(pred).astype(int)})
path = os.path.join(OUT_DIR, 'knn_submission.csv')
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
submission.head()

wrote ../outputs/knn_submission.csv (5000, 2)


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,2
4,9004,1


## How to improve later
- Tune `k` and `weights='distance'`.
- Reduce dimensionality first (PCA) to fight the curse of dimensionality and decorrelate features.
- Drop redundant correlated features before computing distances.